In [1]:
!pip install sentence-transformers

In [2]:
import json
import os
from tqdm import tqdm
from sentence_transformers import SentenceTransformer



In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
CHUNK_JSON_FILE = "laborlaw_chunks.json"
EMBEDDING_JSON_FILE = "laborlaw_embeddings.json"

GG_COLAB = "/content/drive/MyDrive/"
INPUT_FILE = f'{GG_COLAB}/laborlaw/{CHUNK_JSON_FILE}'
OUTPUT_FILE = f'{GG_COLAB}/laborlaw/{EMBEDDING_JSON_FILE}'

In [5]:
MODEL_NAME = "BAAI/bge-m3"
BATCH_SIZE = 32

In [6]:
def load_chunks(path):
  #load chunks từ file json
  with open(path, "r", encoding="utf-8") as f:
    data = json.load(f)
  if data: print('load json thành công!')
  else:
    print('load json fail!')

  chunks = data["chunks"]
  print(f'=> đã load {len(chunks)} chunk')
  return chunks


def embedding_chunks(chunks, model_name=MODEL_NAME, batch_size=BATCH_SIZE):
  # tạo embedding cho từng chunk

  model = SentenceTransformer(model_name)
  print(f'Load model {model_name}')

  # lấy text cần embed
  texts = []
  for chunk in chunks:
    texts.append(chunk["content_with_context"])

  all_embeddings = []
  for i in tqdm(range(0, len(texts), batch_size), desc="Embedding"):
    batch = texts[i:i + batch_size] # lấy ptu thứ i đến i + batch size
    embeddings = model.encode(batch, normalize_embeddings=True)
    for emb in embeddings:
      all_embeddings.append(emb.tolist())
  # gắn embedding vào mỗi chunk
  for chunk, emb in zip(chunks, all_embeddings):
    chunk["embedding"] = emb
  return chunks

def save_embedding(chunks,path):
  with open(path, "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

  print(f'Đã lưu {len(chunks)} chunk với emb vào {path}')

def run_embedding():
  # chunk-> emb-> file json
  chunks = load_chunks(INPUT_FILE)
  chunks_with_emb = embedding_chunks(chunks, MODEL_NAME, BATCH_SIZE)
  save_embedding(chunks_with_emb,OUTPUT_FILE)

run_embedding()

load json thành công!
=> đã load 684 chunk


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Load model BAAI/bge-m3


Embedding: 100%|██████████| 22/22 [00:24<00:00,  1.09s/it]


Đã lưu 684 chunk với emb vào /content/drive/MyDrive//laborlaw/laborlaw_embeddings.json
